In [ ]:
from sklearn.externals import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

In [ ]:
# Load models and other resources
rf = joblib.load('models/rf_model.pkl') # Random Forest model
lr = joblib.load('models/lr_model.pkl') # logistic regression model
bert = joblib.load('models/bert_model.pkl') # BERT model

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import pandas as pd


def flesch_reading_ease(text):
    """
    Calcule le score de lisibilité de Flesch Reading Ease pour un texte anglais.
    """
    # Nombre de phrases (approximation simple)
    sentences = re.split(r'[.!?]+', text)
    sentences = [s for s in sentences if s.strip()]
    n_sentences = max(len(sentences), 1)
    # Nombre de mots
    words = re.findall(r'\w+', text)
    n_words = max(len(words), 1)

    # Nombre de syllabes
    def count_syllables(word):
        word = word.lower()
        vowels = "aeiouy"
        count = 0
        prev_char_was_vowel = False
        for char in word:
            if char in vowels:
                if not prev_char_was_vowel:
                    count += 1
                prev_char_was_vowel = True
            else:
                prev_char_was_vowel = False
        if word.endswith("e"):
            count = max(1, count - 1)
        return max(count, 1)
    
    
    n_syllables = sum(count_syllables(word) for word in words)
    # Formule Flesch Reading Ease
    score = 206.835 - 1.015 * (n_words / n_sentences) - 84.6 * (n_syllables / n_words)
    return round(score, 4)


In [ ]:
def flesch_kincaid_grade(text):
    """
    Calcule le score de niveau scolaire Flesch-Kincaid Grade pour un texte anglais.
    """
    # Nombre de phrases (approximation simple)
    sentences = re.split(r'[.!?]+', text)
    sentences = [s for s in sentences if s.strip()]
    n_sentences = max(len(sentences), 1)
    # Nombre de mots
    words = re.findall(r'\w+', text)
    n_words = max(len(words), 1)

    # Nombre de syllabes
    def count_syllables(word):
        word = word.lower()
        vowels = "aeiouy"
        count = 0
        prev_char_was_vowel = False
        for char in word:
            if char in vowels:
                if not prev_char_was_vowel:
                    count += 1
                prev_char_was_vowel = True
            else:
                prev_char_was_vowel = False
        if word.endswith("e"):
            count = max(1, count - 1)
        return max(count, 1)
    
    n_syllables = sum(count_syllables(word) for word in words)
    # Formule Flesch-Kincaid Grade
    grade = 0.39 * (n_words / n_sentences) + 11.8 * (n_syllables / n_words) - 15.59
    return round(grade, 4)


In [ ]:
def preprocess_text(text):
    """Nettoie et préprocess le texte sans dépendre de word_tokenize (évite l'erreur punkt_tab)"""
    if pd.isna(text):
        return ""
    
    # Convertir en minuscules
    text = text.lower()
    
    # Supprimer les caractères spéciaux
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenisation simple (split sur les espaces)
    tokens = text.split()
    
    # Supprimer les stop words
    stop_words = set(stopwords.words('english'))  # ou 'french'
    tokens = [token for token in tokens if token not in stop_words]
    
    # Stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(token) for token in tokens]
    
    return ' '.join(tokens)

In [ ]:
def predict_fake_news(text, model_type='rf'):
    """
    Prédit si un article est faux ou vrai

    Args:
        text: Texte de l'article
        model_type: 'rf' (Random Forest) ou 'bert' ou 'lr' (Logistic Regression)

    Returns:
        dict: {'prediction': 'REAL'/'FAKE', 'confidence': float}
    """

    if model_type == 'rf':
        # Extraire les features
        processed_text = preprocess_text(text)
        features = {
            'text_length': len(text),
            'word_count': len(text.split()),
            'flesch_score': flesch_reading_ease(text),
            'flesch_kincaid': flesch_kincaid_grade(text),
            'caps_ratio': sum(1 for c in text if c.isupper()) / len(text) if len(text) > 0 else 0,
            'exclamation_count': text.count('!'),
            'question_count': text.count('?')
        }

        # Utiliser un DataFrame pour conserver les noms de features et éviter le warning
        import pandas as pd
        X_pred = pd.DataFrame([features])
        prediction = rf_model.predict(X_pred)[0]
        confidence = rf_model.predict_proba(X_pred)[0].max()

    elif model_type == 'lr':
        processed_text = preprocess_text(text)
        X_pred = tfidf.transform([processed_text])
        prediction = lr_model.predict(X_pred)[0]
        confidence = lr_model.predict_proba(X_pred)[0].max()

    elif model_type == 'bert':
        # Tokenisation
        inputs = tokenizer(text, return_tensors='pt', truncation=True,
                          padding=True, max_length=512)

        # Prédiction
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
            confidence = predictions.max().item()
            prediction = predictions.argmax().item()

    result = {
        'prediction': 'REAL' if prediction == 1 else 'FAKE',
        'confidence_score': round(confidence * 100, 2),
        'authenticity_percentage': round(confidence * 100 if prediction == 1 else (1-confidence) * 100, 2)
    }

    return result

In [ ]:
article_test = """This is a sample article to test the readability scores. It contains several sentences and words to analyze the Flesch Reading Ease and Flesch-Kincaid Grade scores. The goal is to ensure that the functions work correctly and return appropriate values based on the text provided."""

In [ ]:
result = predict_fake_news(article_test, model_type='lr')
print(f"Prédiction: {result['prediction']}")
print(f"Score d'authenticité: {result['authenticity_percentage']}%")